In [58]:
import pandas as pd
import numpy as np
import os
print("Libraries successfully imported!")

Libraries successfully imported!


In [59]:
matches=pd.read_csv("../data/processed/epl_matches_clean.csv")
matches.dtypes
matches["Date"]=pd.to_datetime(matches["Date"], errors="coerce")
matches.dtypes
matches = matches.dropna(
    subset=["Date", "HomeTeam", "AwayTeam", "HomeGoals", "AwayGoals", "MatchResult"]
).copy()
matches=matches.sort_values("Date").reset_index(drop=True)
matches["MatchID"]=range(len(matches))
matches.head()


,Unnamed: 0.1,Unnamed: 0,MatchWeek,Date,HomeTeam,FullTimeScore,AwayTeam,Source_File,HomeGoals,AwayGoals,MatchResult,TotalGoals,GoalDifference,Year,Month,MatchID
0,0,0,1,2010-08-14,Bolton Wanderers FC,0-0,Fulham FC,epl_2010_11,0,0,D,0,0,2010,8,0
1,1,1,1,2010-08-14,Wigan Athletic FC,0-4,Blackpool FC,epl_2010_11,0,4,AW,4,-4,2010,8,1
2,2,2,1,2010-08-14,Aston Villa FC,3-0,West Ham United FC,epl_2010_11,3,0,HW,3,3,2010,8,2
3,3,3,1,2010-08-14,Wolverhampton Wanderers FC,2-1,Stoke City FC,epl_2010_11,2,1,HW,3,1,2010,8,3
4,4,4,1,2010-08-14,Sunderland AFC,2-2,Birmingham City FC,epl_2010_11,2,2,D,4,0,2010,8,4


In [60]:
# Convert matches into team history
# homeTeam
home_history=matches[["MatchID", "Date", "HomeTeam", "AwayTeam", "HomeGoals", "AwayGoals"]].copy()
home_history.columns=["MatchID", "Date", "Team", "Opponent", "GoalsFor", "GoalsAgainst"]
# awayTeam
away_history=matches[["MatchID", "Date", "AwayTeam", "HomeTeam", "AwayGoals", "HomeGoals"]].copy()
away_history.columns=["MatchID", "Date", "Team", "Opponent", "GoalsFor", "GoalsAgainst"]
#combining
team_history=pd.concat(
    [home_history, away_history],
    ignore_index=True
)
#match points
team_history["Points"]=np.select(
    [
        team_history["GoalsFor"]>team_history["GoalsAgainst"],
        team_history["GoalsFor"]==team_history["GoalsAgainst"]
    ],
    [3, 1],
    default=0
)
team_history["Win"]=(team_history["GoalsFor"]>team_history["GoalsAgainst"]).astype(int)
team_history["Draw"]=(team_history["GoalsFor"]==team_history["GoalsAgainst"]).astype(int)
team_history["Loss"]=(team_history["GoalsFor"]<team_history["GoalsAgainst"]).astype(int)
#Sort each teams matches
team_history=team_history.sort_values(["Team", "Date", "MatchID"]).reset_index(drop=True)
team_history.head(10)

,MatchID,Date,Team,Opponent,GoalsFor,GoalsAgainst,Points,Win,Draw,Loss
0,1901,2015-08-08,AFC Bournemouth,Aston Villa FC,0,1,0,0,0,1
1,1919,2015-08-17,AFC Bournemouth,Liverpool FC,0,1,0,0,0,1
2,1920,2015-08-22,AFC Bournemouth,West Ham United FC,4,3,3,1,0,0
3,1933,2015-08-29,AFC Bournemouth,Leicester City FC,1,1,1,0,1,0
4,1944,2015-09-12,AFC Bournemouth,Norwich City FC,1,3,0,0,0,1
5,1953,2015-09-19,AFC Bournemouth,Sunderland AFC,2,0,3,1,0,0
6,1966,2015-09-26,AFC Bournemouth,Stoke City FC,1,2,0,0,0,1
7,1971,2015-10-03,AFC Bournemouth,Watford FC,1,1,1,0,1,0
8,1981,2015-10-17,AFC Bournemouth,Manchester City FC,1,5,0,0,0,1
9,1996,2015-10-25,AFC Bournemouth,Tottenham Hotspur FC,1,5,0,0,0,1


In [61]:
# Columns we want to calculate historical averages for
history_columns = [
    "Points",
    "Win",
    "GoalsFor",
    "GoalsAgainst"
]
# Calculate features using only previous matches
for column in history_columns:
    team_history[f"Recent{column}"] = (
        team_history
        .groupby("Team")[column]
        .transform(
            lambda s: s.shift(1).rolling(
                window=5,
                min_periods=1
            ).mean()
        )
    )
# Rename features to clearer names
team_history = team_history.rename(columns={
    "RecentPoints": "RecentPoints",
    "RecentWin": "RecentWinRate",
    "RecentGoalsFor": "RecentGoalsScored",
    "RecentGoalsAgainst": "RecentGoalsConceded"
})
# Calculate recent goal difference
team_history["RecentGoalDifference"] = (
    team_history["RecentGoalsScored"]
    - team_history["RecentGoalsConceded"]
)
team_history[
    [
        "Team",
        "Date",
        "Points",
        "RecentPoints",
        "RecentWinRate",
        "RecentGoalsScored",
        "RecentGoalsConceded",
        "RecentGoalDifference"
    ]
].tail(15)

,Team,Date,Points,RecentPoints,RecentWinRate,RecentGoalsScored,RecentGoalsConceded,RecentGoalDifference
8321,Wolves,2021-02-14,3,1.0,0.2,0.8,1.0,-0.2
8322,Wolves,2021-02-19,3,1.6,0.4,0.8,0.6,0.2
8323,Wolves,2021-02-27,1,2.0,0.6,1.0,0.6,0.4
8324,Wolves,2021-03-02,0,2.2,0.6,1.2,0.6,0.6
8325,Wolves,2021-03-06,1,1.6,0.4,1.0,1.2,-0.2
8326,Wolves,2021-03-15,0,1.6,0.4,1.0,1.2,-0.2
8327,Wolves,2021-04-05,0,1.0,0.2,0.6,1.2,-0.6
8328,Wolves,2021-04-09,3,0.4,0.0,0.8,1.8,-1.0
8329,Wolves,2021-04-17,3,0.8,0.2,0.8,1.6,-0.8
8330,Wolves,2021-04-25,0,1.4,0.4,0.8,0.8,0.0


In [62]:
#No of matches played before current match
team_history["MatchesPlayedBefore"]=(team_history.groupby("Team").cumcount())
#Days since last match
team_history["DaysSinceLastMatch"]=(team_history.groupby("Team")["Date"].diff().dt.days)
team_history[["Team", "Date", "MatchesPlayedBefore", "DaysSinceLastMatch"]].tail(5)

,Team,Date,MatchesPlayedBefore,DaysSinceLastMatch
8331,Wolves,2021-05-03,33,8.0
8332,Wolves,2021-05-09,34,6.0
8333,Wolves,2021-05-16,35,7.0
8334,Wolves,2021-05-19,36,3.0
8335,Wolves,2021-05-23,37,4.0


In [69]:
# Features to extract for each team
feature_columns = [
    "RecentPoints",
    "RecentWinRate",
    "RecentGoalsScored",
    "RecentGoalsConceded",
    "RecentGoalDifference",
    "MatchesPlayedBefore",
    "DaysSinceLastMatch"
]

# Extract home team features
home_features = team_history[
    ["MatchID"] + feature_columns
].copy()

home_features = home_features.rename(columns={
    column: f"Home{column}"
    for column in feature_columns
})

# Extract away team features
away_features = team_history[
    ["MatchID"] + feature_columns
].copy()

away_features = away_features.rename(columns={
    column: f"Away{column}"
    for column in feature_columns
})

# Merge features into the original match dataset
feature_matches = matches.merge(
    home_features,
    on="MatchID",
    how="left"
)

feature_matches = feature_matches.merge(
    away_features,
    on="MatchID",
    how="left"
)
print("Feature dataset shape:", feature_matches.shape)
feature_matches.head()

Feature dataset shape: (16672, 30)


,Unnamed: 0.1,Unnamed: 0,MatchWeek,Date,HomeTeam,FullTimeScore,AwayTeam,Source_File,HomeGoals,AwayGoals,...,HomeRecentGoalDifference,HomeMatchesPlayedBefore,HomeDaysSinceLastMatch,AwayRecentPoints,AwayRecentWinRate,AwayRecentGoalsScored,AwayRecentGoalsConceded,AwayRecentGoalDifference,AwayMatchesPlayedBefore,AwayDaysSinceLastMatch
0,0,0,1,2010-08-14,Bolton Wanderers FC,0-0,Fulham FC,epl_2010_11,0,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
1,0,0,1,2010-08-14,Bolton Wanderers FC,0-0,Fulham FC,epl_2010_11,0,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
2,0,0,1,2010-08-14,Bolton Wanderers FC,0-0,Fulham FC,epl_2010_11,0,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
3,0,0,1,2010-08-14,Bolton Wanderers FC,0-0,Fulham FC,epl_2010_11,0,0,...,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN
4,1,1,1,2010-08-14,Wigan Athletic FC,0-4,Blackpool FC,epl_2010_11,0,4,...,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN


In [70]:
#Check data leakage
leakage_columns=["HomeGoals", "AwayGoals", "FullTimeScore", "TotalGoals", "GoalDifference", "MatchResult"]
existing_leakage_columns=[
    column for column in leakage_columns
    if column in feature_matches.columns
]
print(existing_leakage_columns)
history_features=[column for column in feature_matches.columns
                  if "Recent" in column
                  or "MatchesPlayedBefore" in column
                  or "DaysSinceLastMatch" in column]
print(history_features)
#Check missing values
feature_matches[history_features].isnull().sum()

['HomeGoals', 'AwayGoals', 'FullTimeScore', 'TotalGoals', 'GoalDifference', 'MatchResult']
['HomeRecentPoints', 'HomeRecentWinRate', 'HomeRecentGoalsScored', 'HomeRecentGoalsConceded', 'HomeRecentGoalDifference', 'HomeMatchesPlayedBefore', 'HomeDaysSinceLastMatch', 'AwayRecentPoints', 'AwayRecentWinRate', 'AwayRecentGoalsScored', 'AwayRecentGoalsConceded', 'AwayRecentGoalDifference', 'AwayMatchesPlayedBefore', 'AwayDaysSinceLastMatch']


HomeRecentPoints            112
HomeRecentWinRate           112
HomeRecentGoalsScored       112
HomeRecentGoalsConceded     112
HomeRecentGoalDifference    112
HomeMatchesPlayedBefore       0
HomeDaysSinceLastMatch      112
AwayRecentPoints            112
AwayRecentWinRate           112
AwayRecentGoalsScored       112
AwayRecentGoalsConceded     112
AwayRecentGoalDifference    112
AwayMatchesPlayedBefore       0
AwayDaysSinceLastMatch      112
dtype: int64

In [71]:
#Save engineered features
os.makedirs("../data/processed", exist_ok=True)
feature_matches.to_csv("../data/processed/epl_matches_features.csv")
print("Feature dataset saved successfully!")

Feature dataset saved successfully!
